In [ ]:
import scanpy as sc
import pandas as pd
import memento
import numpy as np
import os

In [ ]:
data_path = '/home3/ciervo/scMULTIOME/Analisi/scRNA/memento/'

In [ ]:
# Load matrix
adata = sc.read_mtx(data_path + 'count_mtx.mtx')

# Load metadata
metadata = pd.read_csv(data_path + 'metadata.csv')
adata.obs = metadata

# Load genes
genes = pd.read_csv(data_path + 'genes.csv')
adata.obs_names = adata.obs['Barcode']
adata.var_names = genes['gene']


In [ ]:
for i in range(1, 8):

    mp_label = f"MP_{i}"
    print(f"\nProcessing {mp_label}")

    # Subset for current MP
    adata_subset = adata[adata.obs["Metaprogram_assignment"] == mp_label].copy()

    # Skip if no cells
    if adata_subset.n_obs == 0:
        print(f"{mp_label} has no cells — skipping")
        continue

    # Encode Responder as binary
    adata_subset.obs['Responder'] = (
        adata_subset.obs['Responder']
        .apply(lambda x: 1 if x == 'R' else 0)
        .astype(int)
    )

    # Check both classes exist
    if adata_subset.obs['Responder'].nunique() < 2:
        print(f"{mp_label} has only one Responder class — skipping")
        continue

    print(f"Running memento on {mp_label}")

    result_1d = memento.binary_test_1d(
        adata=adata_subset,
        capture_rate=0.07,
        treatment_col='Responder',
        num_cpus=5,
        num_boot=5000
    )

    output_file = os.path.join(data_path, f"{mp_label}_RvsNR.csv")
    result_1d.to_csv(output_file)

    print(f"Saved results to {output_file}")
